In [14]:
from data_utils import HarmonicGraphDataset, harmonic_graph_collate_fn
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
import torch
from torch.utils.data import DataLoader

from models_graph import HarmonicGraphEncoder
from generate_utils import load_AttnFiLMSEModel

import pickle

In [15]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

graph_model = HarmonicGraphEncoder()
graph_model.to(device)

# load the model
model = load_AttnFiLMSEModel(
    tokenizer=tokenizer,
    guidance_dim=graph_model.output_dim
)

In [17]:
dataset = HarmonicGraphDataset(gjt_train, tokenizer, model)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=harmonic_graph_collate_fn)

In [ ]:
batch = next(iter(dataloader)) # TODO: check idx: 246 - bar_start: 15 bar_end: 16
print(batch.keys())

idx: 217
bar_start: 8 bar_end: 11
idx: 468
bar_start: 7 bar_end: 11
idx: 235
bar_start: 15 bar_end: 16
idx: 74
bar_start: 10 bar_end: 11
idx: 191
bar_start: 4 bar_end: 7
idx: 246
bar_start: 15 bar_end: 16


AttributeError: 'Chord' object has no attribute 'pitch_classes'

In [ ]:
logits = model(
    batch['pianoroll'].to(model.device),
    batch['real_harmony_ids'].to(model.device), # TODO: masked version - DONE: 'mask_token_positions' bool
    batch['real_graph'].to(model.device)
)
# TODO: when training with graph guidance, we can train for all graph segment
# masked tokens in parallel

TypeError: linear(): argument 'input' (position 1) must be Tensor, not HeteroDataBatch